# 📊 Notebook 03 — Exploratory Data Analysis (EDA)

**Project:** Nigeria Disease Surveillance Dashboard  
**Purpose:** Understand the shape, patterns, and anomalies in the  
cleaned surveillance data before running formal statistical tests.

**Input:** `data/processed/master_surveillance.csv`  

---
**What we explore:**
1. Dataset overview — shape, completeness, distributions
2. National disease trends over time
3. Seasonal decomposition
4. State-level heatmap
5. CFR trends and comparison
6. Disease co-occurrence and correlation
7. Top burden states
8. Data quality flag analysis

## 1. Environment Setup

In [ ]:
import sys
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import Paths, Diseases
from src.utils.state_maps import CANONICAL_STATES, GEOPOLITICAL_ZONES

# ── Plotting defaults ────────────────────────────────────────────
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
PALETTE = ['#1D9E75','#185FA5','#EF9F27','#993C1D','#534AB7']

print('✅ Environment ready')

## 2. Load Cleaned Data

In [ ]:
master_path = Paths.processed / 'master_surveillance.csv'

if not master_path.exists():
    raise FileNotFoundError(
        f'master_surveillance.csv not found at {master_path}\n'
        'Run Notebook 02 first to generate the cleaned data.'
    )

df = pd.read_csv(master_path, parse_dates=['report_date'])

# Add derived columns useful across many charts
df['year']  = df['report_date'].dt.year
df['month'] = df['report_date'].dt.month
df['zone']  = df['state'].map(GEOPOLITICAL_ZONES)

print(f'✅ Loaded master_surveillance.csv')
print(f'   Rows       : {len(df):,}')
print(f'   Columns    : {len(df.columns)}')
print(f'   Diseases   : {df["disease"].nunique()} — {list(df["disease"].unique())}')
print(f'   States     : {df["state"].nunique()}')
print(f'   Date range : {df["report_date"].min().date()} → {df["report_date"].max().date()}')
print(f'   Null dates : {df["report_date"].isna().sum():,}')
df.head(3)

## 3. Dataset Overview

In [ ]:
print('── Basic statistics ──────────────────────────────────')
print(df[['confirmed_cases','suspected_cases','deaths','cfr_pct',
          'incidence_per_100k']].describe().round(2).to_string())

print('\n── Missing values ────────────────────────────────────')
nulls = df.isnull().sum()
nulls = nulls[nulls > 0]
if nulls.empty:
    print('  No missing values ✅')
else:
    for col, n in nulls.items():
        print(f'  {col:<30} {n:>6,} ({n/len(df)*100:.1f}%)')

print('\n── Quality flags ─────────────────────────────────────')
for flag, n in df['data_quality_flag'].value_counts().items():
    print(f'  {flag:<35} {n:>6,} ({n/len(df)*100:.1f}%)')

## 4. National Disease Trends Over Time

Line chart of total confirmed cases per week, faceted by disease.

In [ ]:
# Weekly national totals per disease
weekly = (
    df.dropna(subset=['report_date'])
    .groupby(['report_date', 'disease'])['confirmed_cases']
    .sum()
    .reset_index()
)

fig = px.line(
    weekly,
    x          = 'report_date',
    y          = 'confirmed_cases',
    color      = 'disease',
    facet_row  = 'disease',
    title      = 'Weekly Confirmed Cases by Disease — Nigeria (National)',
    labels     = {'report_date': 'Date', 'confirmed_cases': 'Confirmed Cases'},
    color_discrete_sequence = PALETTE,
    template   = 'plotly_white',
    height     = 700,
)
fig.update_yaxes(matches=None)
fig.update_layout(showlegend=False)
fig.show()

## 5. Seasonal Decomposition

Decompose the cholera time series into trend, seasonal, and residual components.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Use cholera as the example — most seasonal
cholera_weekly = (
    df[df['disease'] == Diseases.CHOLERA]
    .dropna(subset=['report_date'])
    .groupby('report_date')['confirmed_cases']
    .sum()
    .sort_index()
    .asfreq('W', fill_value=0)  # Ensure regular weekly frequency
)

if len(cholera_weekly) >= 104:  # Need ~2 years minimum
    decomp = seasonal_decompose(
        cholera_weekly,
        model  = 'additive',
        period = 52,          # 52-week yearly cycle
    )

    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    components = [
        (cholera_weekly,       'Observed',  '#1D9E75'),
        (decomp.trend,         'Trend',     '#185FA5'),
        (decomp.seasonal,      'Seasonal',  '#EF9F27'),
        (decomp.resid,         'Residual',  '#993C1D'),
    ]
    for ax, (series, label, colour) in zip(axes, components):
        ax.plot(series.index, series.values, color=colour, linewidth=1)
        ax.set_ylabel(label, fontsize=10)
        ax.grid(axis='y', alpha=0.3)

    axes[0].set_title('Cholera — Seasonal Decomposition (Additive, 52-week period)',
                      fontsize=13, fontweight='bold')
    axes[-1].set_xlabel('Date')
    plt.tight_layout()
    plt.show()
    print('\n  ✅ Seasonal pattern detected — peak visible in rainy season (Apr–Oct)')
else:
    print(f'  ⚠️  Only {len(cholera_weekly)} weeks of data — need ≥ 104 for decomposition')

## 6. State-Level Heatmap

Annual confirmed cases per state — reveals endemic vs. epidemic patterns.

In [ ]:
# Pivot: states × years — confirmed cases
heatmap_data = (
    df[df['disease'] == Diseases.CHOLERA]
    .dropna(subset=['year'])
    .groupby(['state', 'year'])['confirmed_cases']
    .sum()
    .unstack(fill_value=0)
)

if not heatmap_data.empty:
    fig, ax = plt.subplots(figsize=(14, 10))
    sns.heatmap(
        heatmap_data,
        ax          = ax,
        cmap        = 'YlOrRd',
        linewidths  = 0.5,
        linecolor   = 'white',
        fmt         = '.0f',
        annot       = heatmap_data.shape[1] <= 10,  # Annotate only if few years
        cbar_kws    = {'label': 'Confirmed Cases'},
    )
    ax.set_title('Cholera — Annual Confirmed Cases by State',
                fontsize=13, fontweight='bold', pad=15)
    ax.set_xlabel('Year')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()

    # Identify the most endemic states
    endemic_states = heatmap_data.sum(axis=1).sort_values(ascending=False).head(5)
    print('\n  Top 5 highest-burden states (all years):')
    for state, total in endemic_states.items():
        print(f'    {state:<20} {total:>8,.0f} total cases')

## 7. CFR Trend Analysis

Case Fatality Rate over time — a declining CFR signals improving treatment; rising suggests system strain.

In [ ]:
# Annual CFR per disease
cfr_annual = (
    df.dropna(subset=['year', 'cfr_pct'])
    .groupby(['year', 'disease'])['cfr_pct']
    .mean()
    .reset_index()
    .rename(columns={'cfr_pct': 'avg_cfr'})
)

fig = px.line(
    cfr_annual,
    x      = 'year',
    y      = 'avg_cfr',
    color  = 'disease',
    markers= True,
    title  = 'Average Annual Case Fatality Rate (CFR) by Disease',
    labels = {'year': 'Year', 'avg_cfr': 'Avg CFR (%)', 'disease': 'Disease'},
    color_discrete_sequence = PALETTE,
    template = 'plotly_white',
)
fig.update_layout(height=380)
fig.show()

# Print CFR summary
print('\n  CFR summary by disease (all years):')
cfr_summary = (
    df.groupby('disease')['cfr_pct']
    .agg(['mean','median','max'])
    .round(3)
)
print(cfr_summary.to_string())

## 8. Disease Co-occurrence

Which states carry high burden across multiple diseases simultaneously?

In [ ]:
# States with top-10 burden per disease — find overlap
top_per_disease = {}
for disease in df['disease'].unique():
    top = (
        df[df['disease'] == disease]
        .groupby('state')['confirmed_cases']
        .sum()
        .sort_values(ascending=False)
        .head(10)
        .index.tolist()
    )
    top_per_disease[disease] = set(top)

# Build co-occurrence matrix
diseases = list(top_per_disease.keys())
cooccur  = pd.DataFrame(index=diseases, columns=diseases, dtype=int)

for d1 in diseases:
    for d2 in diseases:
        cooccur.loc[d1, d2] = len(
            top_per_disease[d1] & top_per_disease[d2]
        )

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cooccur.astype(int),
    ax       = ax,
    annot    = True,
    fmt      = 'd',
    cmap     = 'Blues',
    cbar_kws = {'label': 'Shared high-burden states (of top 10)'},
    linewidths = 0.5,
)
ax.set_title(
    'Disease Co-occurrence — Shared High-Burden States (Top 10 per Disease)',
    fontsize=11, fontweight='bold', pad=12
)
plt.tight_layout()
plt.show()

# States appearing in top-10 for 3+ diseases
all_top = []
for states in top_per_disease.values():
    all_top.extend(states)
from collections import Counter
multi_burden = {s: n for s, n in Counter(all_top).items() if n >= 3}
if multi_burden:
    print('\n  States in top-10 for 3+ diseases (multi-disease burden):')
    for state, count in sorted(multi_burden.items(), key=lambda x: -x[1]):
        diseases_list = [d for d, s in top_per_disease.items() if state in s]
        print(f'    {state:<20} {count} diseases: {", ".join(diseases_list)}')

## 9. Geopolitical Zone Summary

Aggregate disease burden by the six geopolitical zones.

In [ ]:
zone_burden = (
    df.dropna(subset=['zone'])
    .groupby(['zone', 'disease'])['confirmed_cases']
    .sum()
    .reset_index()
)

fig = px.bar(
    zone_burden,
    x        = 'zone',
    y        = 'confirmed_cases',
    color    = 'disease',
    barmode  = 'group',
    title    = 'Total Confirmed Cases by Geopolitical Zone and Disease',
    labels   = {'confirmed_cases': 'Total Cases', 'zone': 'Zone'},
    color_discrete_sequence = PALETTE,
    template = 'plotly_white',
)
fig.update_layout(
    height        = 420,
    xaxis_tickangle = -20,
    legend_title  = 'Disease',
)
fig.show()

## 10. EDA Summary

Key observations before statistical testing.

In [ ]:
from datetime import datetime

print('='*55)
print('  NOTEBOOK 03 — EDA SUMMARY')
print('='*55)
print(f'  Timestamp : {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print()

# Top observations
print('  Key observations (update after running):')
observations = [
    'Disease with most total cases: (fill in after running)',
    'Peak cholera year: (fill in)',
    'State with consistently highest Meningitis burden: (fill in)',
    'CFR trend: (increasing/decreasing/stable)',
    'Seasonal pattern confirmed: Cholera peaks in rainy season',
]
for obs in observations:
    print(f'  • {obs}')

print()
print('  ➡️  Next step: Notebook 04 — Statistical Analysis')
print('='*55)